# Production Career Recommendation Baseline Training Notebook

This notebook provides a structured, reproducible baseline training workflow for the **CareerMapper ML Module** using the **REAL preprocessed IT job roles dataset** (2,561 records).
It implements production-grade data cleansing, class stratification, TF-IDF feature extraction, Random Forest baseline modeling, and model serialization.

In [ ]:
# 1. Import dependencies
import os
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [ ]:
# 2. Define configurations
RAW_DATA_PATH = '../datasets/raw/clean_it_roles_dataset_no_leakage.csv'
MODEL_OUTPUT_PATH = '../models/role_classifier_baseline.joblib'
VEC_OUTPUT_PATH = '../models/role_classifier_baseline_vectorizer.joblib'
EVAL_OUTPUT_PATH = '../evaluation/baseline_report.txt'

### 3. Load Dataset & Handling Missing/Null Values
Loads the actual dataset and cleans null profiles.

In [ ]:
print("Loading dataset...")
df = pd.read_csv(RAW_DATA_PATH)
print(f"Initial raw records loaded: {len(df)}")

# Handle Nulls
df = df.dropna(subset=['text', 'role'])
print(f"Records remaining after dropping nulls: {len(df)}")

### 4. Duplicate Handling
Removes duplicate descriptions to prevent overfitting and evaluation leakage.

In [ ]:
duplicates = df.duplicated(subset=['text']).sum()
print(f"Duplicates found: {duplicates}")

df = df.drop_duplicates(subset=['text'])
total_unique = len(df)
print(f"Unique profiles remaining: {total_unique}")

### 5. Class (Role) Distribution Analysis

In [ ]:
role_counts = df['role'].value_counts()
print("Class Distributions:")
for r, count in role_counts.items():
    percentage = (count / total_unique) * 100
    print(f"  - {r:<30}: {count:<4} ({percentage:.2f}%)")

### 6. NLP Cleaning and Normalization

In [ ]:
def clean_skill_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Preserve technical punctuation like C#, C++, .Net, and GD&T
    text = re.sub(r'[^a-zA-Z0-9&#_+-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Preprocessing descriptions...")
df['cleaned_text'] = df['text'].apply(clean_skill_text)
print("Preprocessing complete!")

### 7. Stratified Train/Test Split
Splits the dataset using class stratification (`stratify=y`) to maintain uniform proportions.

In [ ]:
X = df['cleaned_text']
y = df['role']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training samples  : {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

### 8. TF-IDF Feature Extraction

In [ ]:
print("Fitting TF-IDF Vectorizer...")
vectorizer = TfidfVectorizer(
    token_pattern=r'(?u)\b[a-zA-Z0-9&#_+-]+\b',
    stop_words='english',
    lowercase=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)} terms")
print(f"Train matrix shape: {X_train_tfidf.shape}")

### 9. Fit Baseline Random Forest Model

In [ ]:
print("Training baseline Random Forest...")
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_val_tfidf)
acc = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {acc:.4f}")

### 10. Display Metrics and Save Estimators

In [ ]:
print("\nClassification Report:")
print(classification_report(y_val, y_pred))

# Ensure model folder exists
os.makedirs('../models', exist_ok=True)

joblib.dump(clf, MODEL_OUTPUT_PATH)
joblib.dump(vectorizer, VEC_OUTPUT_PATH)
print(f"Successfully saved models to models/ directory!")